In [ ]:
import os
import sys
from pathlib import Path

# Set these before importing any `reva` modules. Update the placeholder paths
# to match your machine or shared Jupyter environment.
os.environ["REVA_DATA_ROOT"] = "/path/to/your/data/root"
os.environ["REVA_HF_CACHE_ROOT"] = "/path/to/your/hf_cache/root"
os.environ["REVA_CHECKPOINT_ROOT"] = "/path/to/your/checkpoints/root"
os.environ["REVA_EVAL_RESULTS_ROOT"] = "/path/to/your/eval_results/root"
os.environ["REVA_REGION_DATA_ROOT"] = "/path/to/your/region_data/root"
os.environ["REVA_DECONTAMINATION_ROOT"] = "/path/to/your/decontamination/root"
os.environ["REVA_GROUNDING_DINO_ROOT"] = "/path/to/your/groundingdino/root"
os.environ["REVA_VQAV2_ROOT"] = "/path/to/your/vqav2/root"
os.environ["REVA_TEST_IMAGES_ROOT"] = "/path/to/your/test_images/root"

hf_cache_root = Path(os.environ["REVA_HF_CACHE_ROOT"]).expanduser()
os.environ["HF_HOME"] = str(hf_cache_root)
os.environ["HF_HUB_CACHE"] = str(hf_cache_root / "hub")
os.environ["HF_DATASETS_CACHE"] = str(hf_cache_root / "datasets")
os.environ["TRANSFORMERS_CACHE"] = str(hf_cache_root / "hub")

for var_name in (
    "REVA_DATA_ROOT",
    "REVA_HF_CACHE_ROOT",
    "REVA_CHECKPOINT_ROOT",
    "REVA_EVAL_RESULTS_ROOT",
    "REVA_REGION_DATA_ROOT",
    "REVA_DECONTAMINATION_ROOT",
    "REVA_GROUNDING_DINO_ROOT",
    "REVA_VQAV2_ROOT",
    "REVA_TEST_IMAGES_ROOT",
    "HF_HOME",
    "HF_HUB_CACHE",
    "HF_DATASETS_CACHE",
    "TRANSFORMERS_CACHE",
):
    print(f"{var_name} = {os.environ.get(var_name)}")


def find_reva_project_root(start: Path) -> Path:
    override = os.environ.get("REVA_PROJECT_DIR")
    if override:
        return Path(override).expanduser().resolve()

    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "reva" / "evaluation.py").is_file() and (candidate / "reva" / "config.py").is_file():
            return candidate

    raise FileNotFoundError(
        "Could not locate the ReVA project root from the current working directory. "
        "Set REVA_PROJECT_DIR to your cloned repo path."
    )


PROJECT_ROOT = find_reva_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)
print("Working directory:", Path.cwd())


# POPE pHash Decontamination

Remove curriculum samples whose images perceptually overlap POPE evaluation images.

**Reference benchmark:** [RUCAIBox/POPE](https://github.com/RUCAIBox/POPE) — built on **COCO val2014** (500 images × 3 splits: random / popular / adversarial).

**Technique:** pHash Hamming distance ≤ 4 (same as `dataset.py`).

**Curriculum checked:** COCO + RefCOCO/+/g + VG + GRIT (same combined set used for Stage 2 training).

**Outputs:**
- `curriculum_vs_pope_phash.json` — full match log
- `phash_matches_curriculum_pope_report.csv` — CSV audit trail
- `curriculum_pope_clean.pkl` — filtered sample list ready for training

In [ ]:
# Core decontamination + curriculum loaders (run once per environment)
!pip install imagehash pillow "numpy<2.0" tqdm matplotlib pandas pyarrow datasets huggingface_hub img2dataset --quiet

## 0. Download all datasets (run once)

**Total disk ~80–120 GB.** Skips files that already exist (`wget -c`). Optional sections at the bottom (Grounding DINO, VQAv2/GQA) are commented out.

Run the cell below before building the curriculum. Requires network access and may take several hours (GRIT img2dataset is the slowest step).

In [ ]:
%%bash
set -euo pipefail

export DATA_DIR="${REVA_REGION_DATA_ROOT:-$HOME/reva-data/region_data}"
export HF_HOME="${REVA_HF_CACHE_ROOT:-$HOME/reva-data/hf_cache}"
export DECONTAM_DIR="${REVA_DECONTAMINATION_ROOT:-$HOME/reva-data/decontamination}"
export HF_HUB_DISABLE_PROGRESS_BARS=1
export HF_DATASETS_DISABLE_PROGRESS_BARS=1
export TRANSFORMERS_VERBOSITY=error
export TOKENIZERS_PARALLELISM=false

LOG_DIR="${REVA_DOWNLOAD_LOG_ROOT:-$HOME/reva-data/download_logs}"
mkdir -p "$LOG_DIR" "$DATA_DIR/coco" "$DATA_DIR/visual_genome" "$DATA_DIR/grit" "$DECONTAM_DIR/pope_repo/output/coco"

step()     { echo "[$(date +%H:%M:%S)] $1"; }
done_msg() { echo "[$(date +%H:%M:%S)] done: $1"; }
skip_msg() { echo "[$(date +%H:%M:%S)] skip: $1 (already present)"; }

# ------------------------------------------------------------
# 1) COCO train2017 + annotations
# ------------------------------------------------------------
cd "$DATA_DIR/coco"
if [ ! -d train2017 ]; then
  step "COCO train2017 (~19 GB) ..."
  wget -q -c -O train2017.zip http://images.cocodataset.org/zips/train2017.zip
  unzip -qo train2017.zip && rm -f train2017.zip
  done_msg "COCO train2017"
else skip_msg "COCO train2017"; fi

if [ ! -f annotations/instances_train2017.json ]; then
  step "COCO train2017 annotations ..."
  wget -q -c -O annotations_trainval2017.zip http://images.cocodataset.org/annotations/annotations_trainval2017.zip
  unzip -qo annotations_trainval2017.zip && rm -f annotations_trainval2017.zip
  done_msg "COCO annotations"
else skip_msg "COCO annotations"; fi

# ------------------------------------------------------------
# 2) COCO train2014 — RefCOCO
# ------------------------------------------------------------
if [ ! -d train2014 ]; then
  step "COCO train2014 (~13 GB) ..."
  wget -q -c -O train2014.zip http://images.cocodataset.org/zips/train2014.zip
  unzip -qo train2014.zip && rm -f train2014.zip
  done_msg "COCO train2014"
else skip_msg "COCO train2014"; fi

# ------------------------------------------------------------
# 3) COCO val2014 — POPE
# ------------------------------------------------------------
if [ ! -d val2014 ]; then
  step "COCO val2014 (~6 GB) ..."
  wget -q -c -O val2014.zip http://images.cocodataset.org/zips/val2014.zip
  unzip -qo val2014.zip && rm -f val2014.zip
  done_msg "COCO val2014"
else skip_msg "COCO val2014"; fi

# ------------------------------------------------------------
# 4) Visual Genome
# ------------------------------------------------------------
cd "$DATA_DIR/visual_genome"
if [ ! -f region_descriptions.json ]; then
  step "VG region_descriptions.json ..."
  wget -q -c -O region_descriptions.json.zip https://homes.cs.washington.edu/~ranjay/visualgenome/data/dataset/region_descriptions.json.zip
  unzip -qo region_descriptions.json.zip && rm -f region_descriptions.json.zip
  done_msg "VG region_descriptions.json"
else skip_msg "VG region_descriptions.json"; fi

if [ ! -d VG_100K ] || [ -z "$(ls -A VG_100K 2>/dev/null || true)" ]; then
  step "VG_100K images (~9 GB) ..."
  wget -q -c -O images.zip https://cs.stanford.edu/people/rak248/VG_100K_2/images.zip
  unzip -qo images.zip -d VG_100K && rm -f images.zip
  done_msg "VG_100K"
else skip_msg "VG_100K"; fi

if [ ! -d VG_100K_2 ] || [ -z "$(ls -A VG_100K_2 2>/dev/null || true)" ]; then
  step "VG_100K_2 images (~5 GB) ..."
  wget -q -c -O images2.zip https://cs.stanford.edu/people/rak248/VG_100K_2/images2.zip
  unzip -qo images2.zip -d VG_100K_2 && rm -f images2.zip
  done_msg "VG_100K_2"
else skip_msg "VG_100K_2"; fi

# ------------------------------------------------------------
# 5) POPE JSON files
# ------------------------------------------------------------
cd "$DECONTAM_DIR/pope_repo/output/coco"
step "POPE JSON files ..."
wget -q -nc -O coco_pope_random.json https://raw.githubusercontent.com/RUCAIBox/POPE/main/output/coco/coco_pope_random.json
wget -q -nc -O coco_pope_popular.json https://raw.githubusercontent.com/RUCAIBox/POPE/main/output/coco/coco_pope_popular.json
wget -q -nc -O coco_pope_adversarial.json https://raw.githubusercontent.com/RUCAIBox/POPE/main/output/coco/coco_pope_adversarial.json
done_msg "POPE JSON (3 files)"

# ------------------------------------------------------------
# 7) RefCOCO metadata cache
# ------------------------------------------------------------
step "RefCOCO/+/g HuggingFace cache ..."
python3 >> "$LOG_DIR/refcoco_cache.log" 2>&1 << 'PY'
import os
os.environ["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"
from datasets import load_dataset
for repo in ["jxu124/refcoco", "jxu124/refcocog", "jxu124/refcocoplus"]:
    for split in ["train", "val", "testA", "testB"]:
        try:
            load_dataset(repo, split=split)
        except Exception:
            pass
print("refcoco_cache_ok")
PY
done_msg "RefCOCO metadata"

# ------------------------------------------------------------
# 6) GRIT shard 0
# ------------------------------------------------------------
cd "$DATA_DIR/grit"
mkdir -p metadata/grit-20m images

step "GRIT metadata shard 0 ..."
python3 >> "$LOG_DIR/grit_metadata.log" 2>&1 << 'PY'
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
from huggingface_hub import hf_hub_download
from pathlib import Path
hf_hub_download(
    repo_id="zzliang/GRIT",
    repo_type="dataset",
    subfolder="grit-20m",
    filename="coyo_0_snappy.parquet",
    local_dir="$HOME/reva-data/region_data/grit/metadata",
)
PY
done_msg "GRIT metadata"

step "GRIT filter (250K cap) ..."
python3 >> "$LOG_DIR/grit_filter.log" 2>&1 << 'PY'
import pandas as pd
from pathlib import Path
grit_dir = Path("$HOME/reva-data/region_data/grit")
filtered_path = grit_dir / "filtered.parquet"
if not filtered_path.exists():
    df = pd.read_parquet(grit_dir / "metadata/grit-20m/coyo_0_snappy.parquet")
    df = df[df["clip_similarity_vitl14"] >= 0.30]
    df = df[df["noun_chunks"].map(lambda x: x is not None and len(x) > 0)]
    df = df.head(250000).reset_index(drop=True)
    df.to_parquet(filtered_path)
    print(f"rows={len(df)}")
else:
    print("already_exists")
PY
done_msg "GRIT filtered.parquet"

if [ -z "$(find "$DATA_DIR/grit/images" -name '*.jpg' 2>/dev/null | head -1)" ]; then
  step "GRIT img2dataset (verbose log: $LOG_DIR/grit_img2dataset.log) ..."
  img2dataset \
    --url_list "$DATA_DIR/grit/filtered.parquet" \
    --input_format parquet \
    --url_col url \
    --caption_col caption \
    --output_format files \
    --output_folder "$DATA_DIR/grit/images" \
    --processes_count 4 \
    --thread_count 64 \
    --image_size 336 \
    --resize_only_if_bigger True \
    --resize_mode keep_ratio \
    --skip_reencode True \
    --save_additional_columns '["noun_chunks","ref_exps","clip_similarity_vitl14"]' \
    --enable_wandb False \
    >> "$LOG_DIR/grit_img2dataset.log" 2>&1
  done_msg "GRIT images"
else skip_msg "GRIT images"; fi


echo "[$(date +%H:%M:%S)] ALL DATASET DOWNLOADS COMPLETE"


## 1. Setup

In [ ]:
import os
import json
import csv
import pickle
import random
import urllib.request
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
from PIL import Image

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
os.environ['HF_HOME'] = os.environ.get('REVA_HF_CACHE_ROOT') or os.path.expanduser('~/reva-data/hf_cache')

from reva.config import ProjectionAConfig
from reva.dataset import (
    load_coco_detection_samples,
    load_refcoco_samples,
    load_visual_genome_samples,
    load_grit_samples,
    generate_curriculum_decontamination_log,
    filter_curriculum_from_log,
    HAMMING_THRESH,
)

config = ProjectionAConfig()
DECONTAM_DIR = Path(os.environ.get('REVA_DECONTAMINATION_ROOT') or os.path.expanduser('~/reva-data/decontamination'))
DECONTAM_DIR.mkdir(parents=True, exist_ok=True)

print('Config ready.')
print(f'data_dir: {config.data_dir}')
print(f'pHash Hamming threshold: {HAMMING_THRESH}')

## 2. Build combined curriculum dataset

Same loaders as `region_token_pipeline.ipynb`. GRIT download is skipped here if images are already cached under `data_dir/grit/images/`.

In [ ]:
coco_samples = load_coco_detection_samples(config.data_dir)
refcoco_samples = load_refcoco_samples(config.data_dir, splits=['train'])
vg_samples = load_visual_genome_samples(
    config.data_dir, max_per_image=config.vg_max_annotations_per_image
)
grit_samples = load_grit_samples(
    config.data_dir, shard=0,
    max_images=250000,
    max_boxes_per_image=16,
    use_ref_exps=True,
    min_clip_l14=0.30,
    min_box_frac=0.05,
)
vg_samples = vg_samples + grit_samples

all_curriculum = coco_samples + refcoco_samples + vg_samples

print(f'COCO: {len(coco_samples):,}')
print(f'RefCOCO: {len(refcoco_samples):,}')
print(f'VG+GRIT: {len(vg_samples):,}  (GRIT: {len(grit_samples):,})')
print(f'Total: {len(all_curriculum):,} region rows')
print(f'Unique train images: {len({s["image_path"] for s in all_curriculum}):,}')

## 3. Download POPE question files & resolve COCO val2014 paths

POPE ships pre-built JSONL files under `output/coco/` in the GitHub repo. Each line contains an `"image"` field like `COCO_val2014_000000016631.jpg`.

**You need COCO val2014 images on disk** (not train2014 / train2017):
```bash
wget http://images.cocodataset.org/zips/val2014.zip
unzip val2014.zip -d $HOME/reva-data/region_data/coco/
```

In [ ]:
POPE_BASE = DECONTAM_DIR / 'pope_repo'
POPE_JSON_DIR = POPE_BASE / 'output' / 'coco'
POPE_JSON_DIR.mkdir(parents=True, exist_ok=True)

POPE_JSON_URLS = {
    'random': 'https://raw.githubusercontent.com/RUCAIBox/POPE/main/output/coco/coco_pope_random.json',
    'popular': 'https://raw.githubusercontent.com/RUCAIBox/POPE/main/output/coco/coco_pope_popular.json',
    'adversarial': 'https://raw.githubusercontent.com/RUCAIBox/POPE/main/output/coco/coco_pope_adversarial.json',
}

pope_json_paths = []
for split_name, url in POPE_JSON_URLS.items():
    dest = POPE_JSON_DIR / f'coco_pope_{split_name}.json'
    if not dest.exists():
        print(f'Downloading POPE {split_name} ...')
        urllib.request.urlretrieve(url, dest)
    pope_json_paths.append(dest)
    print(f'  {split_name}: {dest} ({dest.stat().st_size / 1e6:.2f} MB)')

In [ ]:
def load_pope_image_filenames(pope_json_paths):
    """Parse POPE JSONL files -> unique COCO val2014 filenames."""
    filenames = set()
    per_split = {}
    for jp in pope_json_paths:
        split_count = 0
        with open(jp) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                item = json.loads(line)
                filenames.add(item['image'])
                split_count += 1
        per_split[jp.name] = split_count
    return filenames, per_split


def collect_pope_reference_paths(pope_json_paths, coco_val2014_dir):
    """
    Map POPE image filenames to local COCO val2014 file paths.
    Returns list of Path objects usable as pHash reference images.
    """
    coco_val2014_dir = Path(coco_val2014_dir)
    filenames, per_split = load_pope_image_filenames(pope_json_paths)

    ref_paths = []
    missing = []
    for fname in sorted(filenames):
        candidate = coco_val2014_dir / fname
        if candidate.exists():
            ref_paths.append(candidate)
        else:
            missing.append(fname)

    print('POPE splits (question rows):', per_split)
    print(f'POPE unique images: {len(filenames):,}')
    print(f'val2014 found on disk: {len(ref_paths):,}')
    print(f'val2014 missing: {len(missing):,}')
    if missing:
        print('First 5 missing:', missing[:5])
        print('Download val2014 if missing count > 0.')
    return ref_paths


# Adjust if your val2014 lives elsewhere
COCO_VAL2014_DIR = config.data_dir / 'coco' / 'val2014'

pope_ref_paths = collect_pope_reference_paths(pope_json_paths, COCO_VAL2014_DIR)
assert len(pope_ref_paths) > 0, (
    f'No POPE reference images found under {COCO_VAL2014_DIR}. '
    'Download COCO val2014 before continuing.'
)

## 4. Clear shared pHash cache (important)

`generate_curriculum_decontamination_log` caches hash matrices at fixed paths under `~/reva-data/`. Delete them before a POPE run so the **reference** matrix is rebuilt from POPE val2014 images (not a previous VQAv2/GQA scan).

In [ ]:
PHASH_CACHE_FILES = [
    Path(os.environ.get('REVA_DATA_ROOT') or os.path.expanduser('~/reva-data')) / 'ref_packed.npy',
    Path(os.environ.get('REVA_DATA_ROOT') or os.path.expanduser('~/reva-data')) / 'train_packed.npy',
    Path(os.environ.get('REVA_DATA_ROOT') or os.path.expanduser('~/reva-data')) / 'train_valid_indices.npy',
]

for p in PHASH_CACHE_FILES:
    if p.exists():
        p.unlink()
        print(f'Deleted cache: {p}')
    else:
        print(f'No cache (ok): {p}')

print('Ready to hash POPE reference + curriculum from scratch.')

## 5. Run pHash decontamination (POPE reference only)

This step hashes all curriculum rows and all POPE val2014 images, then flags pairs with Hamming distance ≤ 4.

In [ ]:
LOG_PATH = DECONTAM_DIR / 'curriculum_vs_pope_phash.json'

log_path = generate_curriculum_decontamination_log(
    curriculum_samples=all_curriculum,
    all_ref_paths=pope_ref_paths,
    config=config,
    output_json_path=str(LOG_PATH),
    type='phash',   # pHash only — no SSCD
)

print(f'Log saved: {log_path}')

## 6. Summarise matches

In [ ]:
with open(log_path) as f:
    log_data = json.load(f)

phash_matches = log_data['phash_matches']
corrupt_indices = set(log_data.get('corrupt_indices', []))
unique_removed = set(m['train_idx'] for m in phash_matches)

print(f'pHash match rows: {len(phash_matches):,}')
print(f'Unique curriculum rows flagged: {len(unique_removed):,} / {len(all_curriculum):,}')
print(f'Corrupt/unreadable rows: {len(corrupt_indices):,}')
print(f'Remaining after pHash purge: {len(all_curriculum) - len(unique_removed):,}')

hamming_dist = Counter(m['hamming_distance'] for m in phash_matches)
print('\nHamming distance distribution:')
for d in sorted(hamming_dist):
    print(f'  distance {d}: {hamming_dist[d]:,}')

In [ ]:
removed_sources = Counter(all_curriculum[idx].get('source', 'unknown') for idx in unique_removed)
total_sources = Counter(s.get('source', 'unknown') for s in all_curriculum)

print(f"{'source':<20} {'total':>10} {'removed':>10} {'remaining':>10} {'% removed':>10}")
print('-' * 65)
for source in sorted(total_sources.keys()):
    total = total_sources[source]
    removed = removed_sources.get(source, 0)
    remaining = total - removed
    pct = removed / total * 100 if total else 0
    print(f'{source:<20} {total:>10,} {removed:>10,} {remaining:>10,} {pct:>9.1f}%')

In [ ]:
# How many unique POPE val2014 images triggered at least one curriculum match
pope_ref_set = set(str(p) for p in pope_ref_paths)
pope_triggered = {m['reference_image'] for m in phash_matches if m['reference_image'] in pope_ref_set}

print(f'POPE reference images on disk:     {len(pope_ref_paths):,}')
print(f'POPE images with ≥1 pHash match: {len(pope_triggered):,}')
print(f'POPE images with no match:         {len(pope_ref_paths) - len(pope_triggered):,}')

## 7. Visual spot-check (optional)

In [ ]:
def show_phash_match(match):
    train_img = Image.open(match['train_image']).convert('RGB')
    ref_img = Image.open(match['reference_image']).convert('RGB')
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(train_img)
    axes[0].set_title(f"Curriculum [{match['train_idx']}]\n{Path(match['train_image']).name}")
    axes[0].axis('off')
    axes[1].imshow(ref_img)
    axes[1].set_title(f"POPE ref (Hamming={match['hamming_distance']})\n{Path(match['reference_image']).name}")
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()


if phash_matches:
    exact = [m for m in phash_matches if m['hamming_distance'] == 0]
    borderline = [m for m in phash_matches if m['hamming_distance'] == HAMMING_THRESH]
    print(f'Exact duplicates (distance=0): {len(exact):,}')
    print(f'Borderline (distance={HAMMING_THRESH}): {len(borderline):,}')
    if exact:
        show_phash_match(random.choice(exact))
    if borderline:
        show_phash_match(random.choice(borderline))
else:
    print('No pHash matches — curriculum is clean w.r.t. POPE.')

## 8. Export CSV audit trail

In [ ]:
csv_path = DECONTAM_DIR / 'phash_matches_curriculum_pope_report.csv'
fieldnames = ['train_idx', 'train_image', 'reference_image', 'hamming_distance', 'train_source']

with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for m in phash_matches:
        writer.writerow({
            'train_idx': m['train_idx'],
            'train_image': m['train_image'],
            'reference_image': m['reference_image'],
            'hamming_distance': m['hamming_distance'],
            'train_source': all_curriculum[m['train_idx']].get('source', 'unknown'),
        })

print(f'Saved {len(phash_matches):,} rows -> {csv_path}')

## 9. Build clean curriculum & save for training

Uses `filter_curriculum_from_log` (pHash matches only; no SSCD threshold applied because we ran `type='phash'`).

In [ ]:
clean_curriculum = filter_curriculum_from_log(
    curriculum_samples=all_curriculum,
    json_log_path=str(log_path),
)

clean_pkl = DECONTAM_DIR / 'curriculum_pope_clean.pkl'
with open(clean_pkl, 'wb') as f:
    pickle.dump(clean_curriculum, f)

print(f'Clean curriculum saved: {clean_pkl}')
print(f'Rows: {len(clean_curriculum):,}')
print(f'Unique images: {len({s["image_path"] for s in clean_curriculum}):,}')

## 10. Load clean curriculum in training notebook

In `region_token_pipeline.ipynb`, replace the sample-loading cell with:

```python
import pickle
from pathlib import Path

with open(Path(os.environ.get('REVA_DECONTAMINATION_ROOT') or os.path.expanduser('~/reva-data/decontamination')) / 'curriculum_pope_clean.pkl', 'rb') as f:
    all_curriculum = pickle.load(f)

# Split back into buckets if needed (optional — train_projection_a accepts flat list via coco/refcoco/vg args)
coco_samples = [s for s in all_curriculum if s.get('source') == 'coco']
refcoco_samples = [s for s in all_curriculum if s.get('source') in ('refcoco', 'refcocog', 'refcoco+')]
vg_samples = [s for s in all_curriculum if s.get('source') in ('visual_genome', 'grit')]
print(f'COCO: {len(coco_samples):,} | RefCOCO: {len(refcoco_samples):,} | VG+GRIT: {len(vg_samples):,}')
```